In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Pharmacist dashboard']")))
    dash_url = driver.current_url
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Customers')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'CRM')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='CRM sections']")))
    print("Dashboard URL:", dash_url)

    # Back then Forward walk the real history (sidebar nav adds no entries -
    # verified: state-only switching, no router in StaffApp.jsx).
    driver.back()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    print("After Back, URL:", driver.current_url)

    driver.forward()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    assert driver.current_url == dash_url, f"Forward did not restore {dash_url!r} (got {driver.current_url!r})."
    assert not driver.find_elements(By.ID, "username"), "Session lost after Browser Forward."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token, "Access token missing after Browser Forward."
    print("After Forward, URL:", driver.current_url)
    print("PASS: Browser Forward navigation works")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("45_browser_forward_FAIL.png")
finally:
    driver.quit()